# Spatial Prisoner's Dilemma — Nowak & May (1992)

**Course:** From Conway to LangGraph — University of Bologna, Dept. of Physics  
**Reference:** Nowak M.A. & May R.M., *Evolutionary games and spatial chaos*, Nature 359, 826–829 (1992)

---

## What this notebook does

We implement the **exact** model from Nowak & May (1992) on a square lattice and reproduce the three-phase diagram:

| Phase | b range | Outcome |
|---|---|---|
| I | 1 < b < 1.8 | Stable coexistence — cooperators and defectors persist indefinitely |
| II | 1.8 ≲ b < 2 | Metastable cooperators — outcome depends on initial conditions |
| III | b ≥ 2 | Defectors dominate — cooperation collapses |

The mean-field theory predicts cooperator extinction for **any b > 1**.  
Spatial structure rescues cooperation in Phase I — the key result.

---

## Payoff matrix (Nowak–May convention)

$$\pi(C,C) = 1 \quad \pi(C,D) = 0 \quad \pi(D,C) = b \quad \pi(D,D) = 0$$

Single free parameter: **b > 1** (benefit of exploiting a cooperator).  
The PD condition 2R > T+S becomes 2 > b, so the interesting range is **1 < b < 2**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
from IPython.display import HTML

# Fix the random seed for reproducibility
np.random.seed(42)

## 1. The Lattice and Neighbourhood

We use a **square lattice** of size L×L with **periodic boundary conditions** (torus topology).  
Each agent i = (row, col) has a **Moore neighbourhood**: the 8 surrounding cells plus itself → 9 interactions per agent.

```
  NW  N  NE
   W  i   E      ← Moore neighbourhood of i (8 neighbours + self)
  SW  S  SE
```

Periodic boundaries mean the lattice wraps around: agent (0,0) has neighbour (L-1, L-1).  
This eliminates edge effects that would otherwise create artificial boundary phases.

Strategies:  `0 = Defect (D)`,  `1 = Cooperate (C)`

In [ ]:
# Moore neighbourhood offsets: 8 neighbours + self (delta_row, delta_col)
MOORE_OFFSETS = [
    (-1, -1), (-1, 0), (-1, 1),
    ( 0, -1), ( 0, 0), ( 0, 1),   # (0,0) = self-interaction
    ( 1, -1), ( 1, 0), ( 1, 1)
]

def make_lattice(L, initial_density=0.5, mode='random'):
    """
    Initialise an L×L lattice of strategies.

    Parameters
    ----------
    L : int
        Lattice side length.
    initial_density : float
        Fraction of cooperators at t=0.
    mode : str
        'random'  — each site is C with probability initial_density
        'single_D' — all C except one central defector (classic Nowak-May IC)
        'single_C' — all D except one central cooperator

    Returns
    -------
    grid : np.ndarray of shape (L, L), dtype int
        1 = Cooperate, 0 = Defect
    """
    if mode == 'random':
        grid = (np.random.rand(L, L) < initial_density).astype(int)
    elif mode == 'single_D':
        # Classic IC used in Nowak & May: all cooperators except one central defector.
        # With b just above 1, this single defector fails to invade — a useful sanity check.
        grid = np.ones((L, L), dtype=int)
        grid[L // 2, L // 2] = 0
    elif mode == 'single_C':
        # All defectors except one central cooperator.
        # With b < 1.8, the cooperator can nucleate a growing cluster.
        grid = np.zeros((L, L), dtype=int)
        grid[L // 2, L // 2] = 1
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return grid

## 2. Payoff Accumulation

**Phase 1 of each generation:** every agent plays against all 9 agents in its Moore neighbourhood (including itself).  
The payoffs use the Nowak–May matrix:

$$\Pi_i = \sum_{j \in \mathcal{N}(i) \cup \{i\}} \pi(s_i, s_j)$$

Since $\pi(C,D) = 0$, $\pi(D,D) = 0$, and $\pi(D,C) = b$, the payoffs simplify:  
- A **cooperator** earns 1 for each cooperator neighbour (including itself), 0 against defectors.  
- A **defector** earns b for each cooperator neighbour, 0 against other defectors.

We implement this efficiently using **np.roll** to shift the grid in each of the 8 directions and sum up neighbour contributions.

In [ ]:
def compute_payoffs(grid, b):
    """
    Compute the accumulated payoff for every agent in one generation.

    Parameters
    ----------
    grid : np.ndarray (L, L), int
        Current strategy configuration.  1=C, 0=D.
    b : float
        Benefit of defecting against a cooperator.  Must be > 1.

    Returns
    -------
    payoffs : np.ndarray (L, L), float
        Π_i for every agent i.

    Notes
    -----
    Vectorised implementation: instead of looping over all (i,j) pairs,
    we use np.roll to shift the strategy grid in each of the 9 offsets
    and accumulate contributions.

    For a focal agent i with strategy s_i:
      - vs cooperating neighbour j (s_j = 1):  π(C,C)=1 if s_i=C,  π(D,C)=b if s_i=D
      - vs defecting neighbour j (s_j = 0):    π(C,D)=0 if s_i=C,  π(D,D)=0 if s_i=D
    So:
      payoff_i += s_j * (1 * s_i  +  b * (1 - s_i))
                = s_j * [s_i + b*(1-s_i)]

    Summing over all j in the Moore neighbourhood gives the total payoff.
    """
    L = grid.shape[0]
    # Count cooperating neighbours (including self) at each site
    coop_neighbours = np.zeros_like(grid, dtype=float)
    for dr, dc in MOORE_OFFSETS:
        # np.roll with periodic boundary: roll by -dr along axis 0, -dc along axis 1
        coop_neighbours += np.roll(np.roll(grid, -dr, axis=0), -dc, axis=1)

    # For cooperators (grid=1): each cooperating neighbour contributes R=1
    # For defectors  (grid=0): each cooperating neighbour contributes T=b
    payoffs = grid * coop_neighbours + (1 - grid) * b * coop_neighbours
    return payoffs

## 3. Strategy Update — Copy-the-Best (Synchronous)

**Phase 2 of each generation:** every agent compares its payoff with those of all 9 agents in its Moore neighbourhood, and **copies the strategy of the highest-scoring agent** (including itself).

$$s_i(t+1) = s_{j^*}(t) \quad \text{where} \quad j^* = \arg\max_{j \in \mathcal{N}(i) \cup \{i\}} \Pi_j(t)$$

**Key points about this rule:**

1. **Synchronous**: all updates use the payoffs computed from the time-t snapshot. No agent sees its neighbours' updated strategies during the update phase.
2. **Deterministic**: no randomness — purely fitness-driven. This is the zero-temperature (κ→0) limit of the Fermi rule.
3. **Ties**: if the agent itself has the maximum payoff (possibly shared), it keeps its current strategy.
4. **Local information only**: each agent can only see 8 neighbours, not the entire lattice.

The synchronous update is what creates the characteristic **period-2 oscillations** visible in Phase I.

In [ ]:
def update_strategies(grid, payoffs):
    """
    Apply one round of Copy-the-Best synchronous update.

    Parameters
    ----------
    grid    : np.ndarray (L, L), int   — current strategies
    payoffs : np.ndarray (L, L), float — payoffs from this generation

    Returns
    -------
    new_grid : np.ndarray (L, L), int  — strategies for next generation

    Algorithm
    ---------
    For each agent i, we need:
      1. The maximum payoff in N(i) ∪ {i}
      2. The strategy of whichever agent achieved that maximum

    We implement this by iterating over the 9 neighbourhood offsets.
    For each offset, we shift both the payoff grid and the strategy grid,
    compare against the running maximum, and update the candidate strategy
    wherever the shifted payoff exceeds the current maximum.

    Tie-breaking: the first offset checked is (0,0) = self, so in a tie
    the agent keeps its own strategy.
    """
    # Initialise: each agent starts by "copying" itself
    best_payoff = payoffs.copy()
    new_grid    = grid.copy()

    for dr, dc in MOORE_OFFSETS:
        if dr == 0 and dc == 0:
            continue  # self already initialised above

        # Shift payoffs and strategies of neighbour (i+dr, j+dc) to position (i,j)
        neighbour_payoff   = np.roll(np.roll(payoffs, -dr, axis=0), -dc, axis=1)
        neighbour_strategy = np.roll(np.roll(grid,    -dr, axis=0), -dc, axis=1)

        # Where the neighbour beats the current best, adopt their strategy
        better = neighbour_payoff > best_payoff
        best_payoff = np.where(better, neighbour_payoff, best_payoff)
        new_grid    = np.where(better, neighbour_strategy, new_grid)

    return new_grid


def step(grid, b):
    """
    One complete generation: payoff accumulation + synchronous strategy update.
    Returns (new_grid, payoffs) where payoffs are from the current generation.
    """
    payoffs  = compute_payoffs(grid, b)
    new_grid = update_strategies(grid, payoffs)
    return new_grid, payoffs

## 4. A Single Run — Visualising the Patterns

We run the model for a fixed value of **b** and visualise the spatial patterns.

**Colour code:**  
- 🔵 Blue = Cooperator (C)  
- 🔴 Red = Defector (D)

Try different values of b to see the three phases:
- `b = 1.5` → Phase I: coexistence with fractal patterns
- `b = 1.85` → Phase II: metastable cooperator islands
- `b = 2.1` → Phase III: defectors take over

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
L          = 100        # lattice side length (L×L agents)
b          = 1.5        # benefit of defection — try 1.5, 1.85, 2.1
T_RUN      = 100        # number of generations to run
PLOT_EVERY = 10         # snapshot frequency for the grid plots

# ── Initialise ────────────────────────────────────────────────────────────────
# Random initial condition: 50% cooperators, 50% defectors
grid = make_lattice(L, initial_density=0.5, mode='random')

# Colour map: 0=D → red, 1=C → blue
cmap = mcolors.ListedColormap(['#C00000', '#2E75B6'])

# ── Run and store snapshots ────────────────────────────────────────────────────
snapshots = [grid.copy()]   # store grids for visualisation
f_C_history = [grid.mean()] # track fraction of cooperators over time

for t in range(T_RUN):
    grid, _ = step(grid, b)
    f_C_history.append(grid.mean())
    if (t + 1) % PLOT_EVERY == 0:
        snapshots.append(grid.copy())

# ── Plot snapshots and f_C trajectory ─────────────────────────────────────────
n_snaps = len(snapshots)
fig, axes = plt.subplots(2, n_snaps // 2 + 1, figsize=(18, 7))
axes = axes.flatten()

for idx, snap in enumerate(snapshots):
    t_snap = 0 if idx == 0 else idx * PLOT_EVERY
    axes[idx].imshow(snap, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
    axes[idx].set_title(f't = {t_snap}', fontsize=10)
    axes[idx].axis('off')

# Use last axis for f_C(t) plot
ax_fc = axes[n_snaps]
ax_fc.plot(f_C_history, color='#2E75B6', linewidth=1.5)
ax_fc.axhline(y=f_C_history[-1], color='gray', linestyle='--', linewidth=0.8)
ax_fc.set_xlabel('Generation', fontsize=10)
ax_fc.set_ylabel('f_C  (fraction cooperators)', fontsize=10)
ax_fc.set_title(f'Cooperator fraction — b = {b}', fontsize=10)
ax_fc.set_ylim(0, 1)
ax_fc.grid(True, alpha=0.3)

# Hide unused axes
for ax in axes[n_snaps + 1:]:
    ax.axis('off')

plt.suptitle(f'Spatial PD — Nowak & May (1992)   b = {b},   L = {L}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"Final cooperator fraction:  f_C = {f_C_history[-1]:.3f}")

## 5. Animation — Watching the Dynamics Unfold

The animation shows the spatial patterns evolving in real time.  
Notice the **characteristic features** depending on b:

- **Phase I (b=1.5):** Rotating kaleidoscopic patterns. Cooperator clusters are stable, with defector tongues unable to penetrate the interior. Period-2 oscillations at the boundaries.
- **Phase II (b=1.85):** Cooperator islands form and slowly erode. Sensitive to initial conditions.
- **Phase III (b=2.1):** Defectors rapidly sweep across the lattice.

In [ ]:
def run_animation(b, L=80, T=80, interval=120):
    """
    Run the spatial PD and return an HTML animation.

    Parameters
    ----------
    b        : float  — benefit parameter
    L        : int    — lattice size
    T        : int    — number of generations to animate
    interval : int    — milliseconds between frames
    """
    grid = make_lattice(L, initial_density=0.5, mode='random')
    cmap = mcolors.ListedColormap(['#C00000', '#2E75B6'])

    fig, (ax_grid, ax_fc) = plt.subplots(1, 2, figsize=(12, 5))
    plt.suptitle(f'Spatial PD — b = {b}   (Nowak & May 1992)', fontsize=12, fontweight='bold')

    # Left panel: lattice
    im = ax_grid.imshow(grid, cmap=cmap, vmin=0, vmax=1, animated=True, interpolation='nearest')
    ax_grid.set_title('Strategy grid   (blue=C, red=D)', fontsize=10)
    ax_grid.axis('off')

    # Right panel: f_C(t)
    f_C_history = [grid.mean()]
    line, = ax_fc.plot([], [], color='#2E75B6', linewidth=1.5)
    ax_fc.set_xlim(0, T)
    ax_fc.set_ylim(0, 1)
    ax_fc.set_xlabel('Generation')
    ax_fc.set_ylabel('f_C')
    ax_fc.set_title('Cooperator fraction', fontsize=10)
    ax_fc.axhline(y=0.5, color='gray', linestyle=':', linewidth=0.8, label='initial f_C')
    ax_fc.grid(True, alpha=0.3)
    gen_text = ax_fc.text(0.02, 0.95, '', transform=ax_fc.transAxes, fontsize=9)

    state = {'grid': grid}

    def animate(frame):
        state['grid'], _ = step(state['grid'], b)
        f_C_history.append(state['grid'].mean())
        im.set_data(state['grid'])
        line.set_data(range(len(f_C_history)), f_C_history)
        gen_text.set_text(f't = {frame + 1}   f_C = {f_C_history[-1]:.3f}')
        return im, line, gen_text

    anim = animation.FuncAnimation(fig, animate, frames=T,
                                   interval=interval, blit=True)
    plt.tight_layout()
    plt.close()
    return HTML(anim.to_jshtml())


# Run animation for Phase I (stable coexistence)
run_animation(b=1.5, L=80, T=80)

## 6. The Phase Diagram — f_C vs b

We now sweep b across the full range [1.0, 2.2] and measure the **asymptotic cooperator fraction** f_C(b).

**What to expect:**
- f_C ≈ constant > 0 for Phase I (1 < b < 1.8)
- Sharp drop around b ≈ 1.8 (Phase I → Phase II transition)
- f_C ≈ 0 for Phase III (b ≥ 2)

**Mean-field prediction:** f_C = 0 for all b > 1 (plotted as dashed line for reference).

This plot is the computational proof that spatial structure qualitatively changes the phase diagram.

In [ ]:
# ── Sweep parameters ──────────────────────────────────────────────────────────
b_values   = np.linspace(1.01, 2.2, 50)   # sweep range
L_phase    = 80                            # lattice size (larger → smoother curve)
T_TRANSIENT = 200                          # discard first T_TRANSIENT generations
T_MEASURE   = 50                           # average over T_MEASURE generations after

f_C_mean = np.zeros(len(b_values))
f_C_std  = np.zeros(len(b_values))

print("Sweeping b values...")
for idx, b_val in enumerate(b_values):
    grid = make_lattice(L_phase, initial_density=0.5, mode='random')

    # Burn-in: let the system reach its attractor
    for _ in range(T_TRANSIENT):
        grid, _ = step(grid, b_val)

    # Measure asymptotic f_C
    f_C_samples = []
    for _ in range(T_MEASURE):
        grid, _ = step(grid, b_val)
        f_C_samples.append(grid.mean())

    f_C_mean[idx] = np.mean(f_C_samples)
    f_C_std[idx]  = np.std(f_C_samples)

    if (idx + 1) % 10 == 0:
        print(f"  b = {b_val:.2f}   f_C = {f_C_mean[idx]:.3f}")

print("Done.")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

# Spatial model result
ax.plot(b_values, f_C_mean, color='#2E75B6', linewidth=2, label='Spatial model  (L=%d)' % L_phase)
ax.fill_between(b_values,
                f_C_mean - f_C_std,
                f_C_mean + f_C_std,
                alpha=0.2, color='#2E75B6', label='±1 std')

# Mean-field prediction: f_C = 0 for all b > 1
ax.axhline(y=0, color='#C00000', linestyle='--', linewidth=1.5, label='Mean-field prediction (f_C = 0)')

# Phase boundaries
ax.axvline(x=1.8, color='gray', linestyle=':', linewidth=1.2)
ax.axvline(x=2.0, color='gray', linestyle=':', linewidth=1.2)
ax.text(1.35, 0.85, 'Phase I\nCoexistence', ha='center', fontsize=9, color='#2E75B6')
ax.text(1.90, 0.85, 'Phase II\nMetastable', ha='center', fontsize=9, color='#E8B931')
ax.text(2.10, 0.85, 'Phase III\nDefectors', ha='center', fontsize=9, color='#C00000')

ax.set_xlabel('Benefit parameter  b', fontsize=12)
ax.set_ylabel('Asymptotic cooperator fraction  f_C', fontsize=12)
ax.set_title('Phase diagram of the Spatial PD — Nowak & May (1992)', fontsize=12, fontweight='bold')
ax.set_xlim(1.0, 2.2)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Update Rule Comparison — Copy-the-Best vs Fermi Rule
## **(need to be fixed !not working !!...exercise..:-)
)**

The Nowak & May model uses **Copy-the-Best** (deterministic, zero temperature).  
The **Fermi rule** (Szabó & Tőke 1998) replaces the hard threshold with a probabilistic one:

$$P(s_i \leftarrow s_j) = \frac{1}{1 + \exp\left(-\frac{\Pi_j - \Pi_i}{\kappa}\right)}$$

- **κ → 0:** step function → Copy-the-Best (zero temperature)
- **κ → ∞:** P → 0.5 regardless of payoffs (random copying, infinite temperature)
- **Finite κ:** stochastic exploration — worse strategies can still spread

The comparison below shows how increasing κ changes the cooperation level and the sharpness of the phase transition.

In [ ]:
def update_fermi(grid, payoffs, kappa):
    """
    Fermi (pairwise comparison) update rule — Szabó & Tőke (1998).

    Each agent i selects one random neighbour j.
    Agent i adopts j's strategy with probability:
        P = 1 / (1 + exp(-(Π_j - Π_i) / κ))

    Parameters
    ----------
    grid    : np.ndarray (L, L), int
    payoffs : np.ndarray (L, L), float
    kappa   : float  — noise / temperature parameter

    Returns
    -------
    new_grid : np.ndarray (L, L), int
    """
    L = grid.shape[0]
    new_grid = grid.copy()

    # Each agent picks a random neighbour (one of the 8 surrounding cells)
    # We vectorise by sampling a random offset index per agent
    neighbour_offsets = [o for o in MOORE_OFFSETS if o != (0, 0)]  # exclude self
    idx = np.random.randint(0, 8, size=(L, L))

    for k, (dr, dc) in enumerate(neighbour_offsets):
        mask = (idx == k)
        nb_payoff   = np.roll(np.roll(payoffs, -dr, axis=0), -dc, axis=1)
        nb_strategy = np.roll(np.roll(grid,    -dr, axis=0), -dc, axis=1)

        # Fermi probability
        delta = nb_payoff - payoffs
        prob  = 1.0 / (1.0 + np.exp(-delta / kappa))

        # Adopt neighbour's strategy with probability prob
        adopt = (np.random.rand(L, L) < prob) & mask
        new_grid = np.where(adopt, nb_strategy, new_grid)

    return new_grid


def run_model(b, L=80, T=250, rule='copy_best', kappa=0.1):
    """
    Run the spatial PD and return the f_C trajectory.

    Parameters
    ----------
    rule : 'copy_best' or 'fermi'
    kappa : float — only used if rule='fermi'
    """
    grid = make_lattice(L, initial_density=0.5, mode='random')
    f_C = [grid.mean()]

    for _ in range(T):
        payoffs = compute_payoffs(grid, b)
        if rule == 'copy_best':
            grid = update_strategies(grid, payoffs)
        else:
            grid = update_fermi(grid, payoffs, kappa)
        f_C.append(grid.mean())

    return np.array(f_C)


# ── Compare the two rules for b = 1.5 (Phase I) ───────────────────────────────
b_test = 1.5
T_comp = 200

np.random.seed(0)
f_copy  = run_model(b_test, T=T_comp, rule='copy_best')
np.random.seed(0)
f_low_k = run_model(b_test, T=T_comp, rule='fermi', kappa=0.05)
np.random.seed(0)
f_mid_k = run_model(b_test, T=T_comp, rule='fermi', kappa=0.5)
np.random.seed(0)
f_hi_k  = run_model(b_test, T=T_comp, rule='fermi', kappa=5.0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(f_copy,  color='#1A1A1A', linewidth=2,   label='Copy-the-Best  (κ → 0)')
ax.plot(f_low_k, color='#2E75B6', linewidth=1.5, label='Fermi  κ = 0.05  (near deterministic)', linestyle='--')
ax.plot(f_mid_k, color='#4A9ECC', linewidth=1.5, label='Fermi  κ = 0.5')
ax.plot(f_hi_k,  color='#E8B931', linewidth=1.5, label='Fermi  κ = 5.0  (high noise)')
ax.set_xlabel('Generation', fontsize=11)
ax.set_ylabel('f_C', fontsize=11)
ax.set_title(f'Update rule comparison — b = {b_test}', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Finite-Size Effects — Does System Size Matter?

The phase boundaries in the Nowak & May diagram are not perfectly sharp — they depend on the **system size L**.  
Near the Phase I/II boundary (b ≈ 1.8), smaller lattices show stronger fluctuations and earlier extinction of cooperators.

This is the spatial analogue of **finite-size scaling** in statistical mechanics:  
the true thermodynamic limit L→∞ is only approached gradually, and the apparent critical point shifts with L.

**Exercise:** observe how the asymptotic f_C near b=1.8 changes as L increases from 30 to 150.

In [ ]:
# ── Finite-size comparison ────────────────────────────────────────────────────
b_range  = np.linspace(1.4, 2.0, 25)
L_values = [30, 60, 100]
T_burn   = 150
T_meas   = 50

results = {}
for L_val in L_values:
    print(f"Running L = {L_val}...")
    f_vals = []
    for b_val in b_range:
        grid = make_lattice(L_val, initial_density=0.5, mode='random')
        for _ in range(T_burn):
            grid, _ = step(grid, b_val)
        samples = []
        for _ in range(T_meas):
            grid, _ = step(grid, b_val)
            samples.append(grid.mean())
        f_vals.append(np.mean(samples))
    results[L_val] = np.array(f_vals)

colors = ['#C00000', '#2E75B6', '#375623']
fig, ax = plt.subplots(figsize=(9, 5))
for (L_val, f_vals), col in zip(results.items(), colors):
    ax.plot(b_range, f_vals, color=col, linewidth=2, label=f'L = {L_val}')

ax.axvline(x=1.8, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('b', fontsize=12)
ax.set_ylabel('Asymptotic f_C', fontsize=12)
ax.set_title('Finite-size effects near the Phase I/II boundary', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Mean-Field vs Spatial — Direct Comparison

We now run the mean-field replicator equation alongside the spatial simulation, starting from the **same initial f_C**, to make the contrast explicit.

The MF replicator equation is:

$$\frac{df_C}{dt} = f_C(1 - f_C)(\langle \Pi_C \rangle - \langle \Pi_D \rangle) = f_C^2(1 - f_C)(1 - b)$$

For b > 1 this is always negative: f_C → 0 monotonically.  
The spatial model (in Phase I) converges to a positive fixed point — the cooperation phase that MF misses entirely.

In [ ]:
from scipy.integrate import odeint

def replicator_mf(f_C, t, b):
    """Mean-field replicator equation for the Nowak-May PD.
    df_C/dt = f_C^2 * (1-f_C) * (1-b)
    For b>1 this is always <= 0: cooperators go extinct.
    """
    return f_C**2 * (1 - f_C) * (1 - b)


b_compare = 1.5      # Phase I
f0        = 0.5      # initial cooperator fraction
T_mf      = 300

# Mean-field trajectory (continuous time, solved via ODE)
t_mf = np.linspace(0, T_mf, 1000)
f_mf = odeint(replicator_mf, f0, t_mf, args=(b_compare,)).flatten()

# Spatial trajectory
np.random.seed(42)
f_spatial = run_model(b_compare, L=100, T=T_mf, rule='copy_best')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_mf,                  f_mf,      color='#C00000', linewidth=2,   label='Mean-field (replicator eq.)', linestyle='--')
ax.plot(range(len(f_spatial)), f_spatial, color='#2E75B6', linewidth=1.5, label='Spatial model (L=100)')
ax.axhline(y=0, color='#C00000', linestyle=':', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Time / Generation', fontsize=11)
ax.set_ylabel('Cooperator fraction  f_C', fontsize=11)
ax.set_title(f'Mean-field vs Spatial — b = {b_compare}  (Phase I)', fontsize=12, fontweight='bold')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotation
ax.annotate('MF predicts\nextinction',
            xy=(200, 0.02), fontsize=9, color='#C00000',
            arrowprops=dict(arrowstyle='->', color='#C00000'),
            xytext=(150, 0.15))
ax.annotate(f'Spatial: f_C → {f_spatial[-1]:.2f}',
            xy=(280, f_spatial[-1]), fontsize=9, color='#2E75B6',
            arrowprops=dict(arrowstyle='->', color='#2E75B6'),
            xytext=(200, f_spatial[-1] + 0.15))

plt.tight_layout()
plt.show()

## 10. Exercises

---

### Exercise 1 — Invasion from a single defector

Start with `mode='single_D'` (one defector in a sea of cooperators) and vary b.

**Questions:**
- For which values of b does the single defector successfully invade?
- What is the invasion threshold? Can you predict it from the payoff matrix?
- What spatial pattern does the invading defector produce?

*Hint:* The defector's payoff advantage depends on how many cooperating neighbours it has at the boundary.

In [ ]:
# ── Exercise 1: single defector invasion ──────────────────────────────────────
# TODO: complete this cell

b_invasion = 1.5   # change this
L_inv      = 80
T_inv      = 80

grid = make_lattice(L_inv, mode='single_D')
cmap = mcolors.ListedColormap(['#C00000', '#2E75B6'])

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
axes[0].imshow(grid, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
axes[0].set_title('t = 0'); axes[0].axis('off')

for t in range(1, 10):
    grid, _ = step(grid, b_invasion)
    axes[t].imshow(grid, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
    axes[t].set_title(f't = {t}'); axes[t].axis('off')

plt.suptitle(f'Single defector invasion — b = {b_invasion}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 2 — Von Neumann vs Moore neighbourhood

The Nowak & May paper uses the **Moore neighbourhood** (8 neighbours + self).  
Modify the code to use the **von Neumann neighbourhood** (4 neighbours + self) and redo the phase diagram.

**Questions:**
- Does the Phase I/II boundary shift? In which direction?
- Why does the coordination number (4 vs 8+self) affect the cooperation threshold?
- Connect to the Ising model: how does the critical temperature depend on coordination number z?

In [ ]:
# ── Exercise 2: von Neumann neighbourhood ─────────────────────────────────────
# TODO: define VON_NEUMANN_OFFSETS and rewrite compute_payoffs / update_strategies
# to accept a neighbourhood parameter.

VON_NEUMANN_OFFSETS = [
    (-1, 0), (0, -1), (0, 0), (0, 1), (1, 0)   # N, W, self, E, S
]

# Your code here...
print("Implement Exercise 2: compare phase diagrams for Moore vs von Neumann neighbourhood.")

### Exercise 3 — Asynchronous update

Nowak & May use **synchronous** update (all agents update simultaneously).  
Implement **asynchronous** update (agents update one at a time in random order).

**Questions:**
- Do the period-2 oscillations disappear?
- Does the asymptotic f_C change significantly?
- Is the phase boundary in the same place?
- Which update scheme is more realistic biologically? Computationally?

In [ ]:
# ── Exercise 3: asynchronous update ───────────────────────────────────────────
# TODO: implement update_async(grid, b) that iterates over agents in random order,
# updating each agent one at a time using the current (partially updated) grid.

def update_async(grid, b):
    """
    Asynchronous Copy-the-Best update.
    Agents are visited in random order; each update is applied immediately
    before the next agent is processed.
    """
    # TODO: implement this
    raise NotImplementedError("Implement asynchronous update")


print("Implement Exercise 3: asynchronous update and compare with synchronous.")

---

## Summary

| Concept | Nowak & May implementation |
|---|---|
| Lattice | L×L square, periodic boundaries |
| Neighbourhood | Moore (8 neighbours + self = 9 interactions) |
| Payoff | C vs C→1, C vs D→0, D vs C→b, D vs D→0 |
| Update rule | Copy-the-Best — deterministic, synchronous |
| Free parameter | b ∈ (1, 2) — benefit of defection |
| Phase I (1 < b < 1.8) | Stable coexistence — spatial structure rescues cooperation |
| Phase II (1.8 ≲ b < 2) | Metastable cooperators — IC-dependent |
| Phase III (b ≥ 2) | Defector dominance — MF and spatial agree |
| Mean-field | Predicts f_C → 0 for **all** b > 1 — qualitatively wrong in Phase I |

**Key physics take-away:**  
Spatial reciprocity — the geometric assortment of cooperators into clusters — acts as the spatial analogue of temporal reciprocity (Grim Trigger, Tit-for-Tat).  
The cluster interior earns payoff 9 (all cooperators); a defector surrounded by other defectors earns 0.  
This payoff asymmetry is what stabilises cooperation against the mean-field prediction.

**Bridge to RL:**  
The Copy-the-Best rule is a greedy policy (ε=0, κ→0).  
The Fermi rule is a Boltzmann/softmax policy at temperature κ.  
Replacing fixed evolutionary rules with Q-learning agents on this lattice is the entry point to multi-agent RL.